# 🎙️ EidCloud Voice Dataset Factory Quickstart

Welcome to **eidcloud-voice-dataset** — an automated Voice Dataset Factory designed for Speech AI, TTS (XTTS / Bark / VITS), and ASR model training.

This 3-cell quickstart demonstrates:
1. **Environment Setup**: Checking PHP 8.2+ and FFmpeg
2. **Synthetic / Sample Audio Generation**: Creating sample voice audio with silences
3. **Dataset Pipeline Execution & Audit**: Transforming raw audio into an LJSpeech-compliant dataset (`metadata.csv`, `train.csv`, `val.csv`, `stats.json`)

In [ ]:
# Cell 1: Environment Setup & Tool Verification
!apt-get update -qq && apt-get install -y php-cli ffmpeg -qq
!php -v
!ffmpeg -version | head -n 1

In [ ]:
# Cell 2: Generate Raw Sample Audio with Voice Silences
import os
import numpy as np
import wave

os.makedirs("raw_audio", exist_ok=True)

def create_synthetic_voice(filename, duration=10.0, sr=22050):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    # Synthetic vocal harmonics: 220Hz fundamental with overtone formants
    signal = (0.6 * np.sin(2 * np.pi * 220 * t) +
              0.3 * np.sin(2 * np.pi * 440 * t) +
              0.15 * np.sin(2 * np.pi * 880 * t))
    
    # Natural conversational pauses (silences)
    pause1 = (t >= 3.0) & (t <= 3.8)
    pause2 = (t >= 7.0) & (t <= 7.8)
    signal[pause1] = 0.001 * np.random.randn(np.sum(pause1))
    signal[pause2] = 0.001 * np.random.randn(np.sum(pause2))
    
    int16_audio = (signal * 32767).astype(np.int16)
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(int16_audio.tobytes())

create_synthetic_voice("raw_audio/speaker1_take01.wav")
create_synthetic_voice("raw_audio/speaker1_take02.wav")
print("✅ Synthetic raw audio files created in raw_audio/")

In [ ]:
# Cell 3: Run eidcloud-voice Build & Audit Pipeline
!php bin/eidcloud-voice build ./raw_audio/ --output=./dataset/ --lufs=-23.0 --min-sec=2.0 --max-sec=10.0 --json
!php bin/eidcloud-voice audit ./dataset/ --strict

print("\n--- Sample metadata.csv (LJSpeech Format: ID|RawText|NormalizedText) ---")
!head -n 5 ./dataset/metadata.csv

print("\n--- Generated Dataset Statistics (stats.json) ---")
!cat ./dataset/stats.json